# Chunking, part 2: dense retrieval, table questions, parent-child (starter)

`03-recall-at-5-bm25.ipynb` showed that with BM25 the three chunkers tie at about 68%
Recall@5, while 94% of answers sit whole in some chunk. This notebook is the starting
point for the three follow-up questions. Every part runs end to end here with a small
CPU model; the `YOUR TURN` cells say what to change.

| Part | Question | Runs here with |
|---|---|---|
| A | Does the title/heading label help **dense** retrieval more than BM25? | `all-MiniLM-L6-v2` (small, CPU) |
| B | On **table** questions only, does Docling beat fixed chunks? | BM25 |
| C | **Parent-child**: search small chunks, hand back the parent section. Does the answer reach the LLM more often? | BM25 |
| D | Export the chosen chunks as `chunks.jsonl` for the retriever track | file in `04-benchmarks/covid-qa/data/chunks/` (gitignored) |

## Capabilities

| Name | What it does | Example |
|---|---|---|
| `docling_chunks` | Docling chunks of one paper, plain or labelled | 43 chunks for one paper |
| `bm25_rank` / `dense_rank` | top-50 chunk indices per question | `[[812, 77, ...], ...]` |
| `hits_at` | 1/0 per question: right paper + gold answer in the top k | `[1, 0, 1, ...]` |
| `report` | paired bootstrap CI between two rows | `+0.4 pts [-0.3, +1.1] tie` |
| `is_table_question` | the gold answer appears inside a table of its paper | `True` |
| `parent_hits` | parent-child Recall@5 | `[1, 1, 0, ...]` |

Offline once the data is on disk; no API key. Part A downloads a small embedding model once.

### Step 1 · Bootstrap the environment

In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()
REPO = _root
print("repository root found")

repository root found


### Step 2 · Load the gold questions (downloads from Hugging Face if missing)

In [2]:
import json
import tarfile

DATA = REPO / "04-benchmarks" / "covid-qa" / "data"
PARSED = DATA / "02_parsed" / "covidqa"
GOLD = DATA / "gold" / "questions.jsonl"
CHUNK_DIR = DATA / "chunks"          # gitignored with the rest of data/


def ensure_data():
    if GOLD.is_file() and PARSED.is_dir():
        return
    import zstandard as zstd
    from huggingface_hub import hf_hub_download
    DATA.mkdir(parents=True, exist_ok=True)
    hf_hub_download("anacodicai-labs/covidqa-bench", "gold/questions.jsonl",
                    repo_type="dataset", local_dir=str(DATA))
    archive = hf_hub_download("anacodicai-labs/covidqa-bench", "parsed_questions_137.tar.zst",
                              repo_type="dataset", local_dir=str(DATA))
    with open(archive, "rb") as fh:
        reader = zstd.ZstdDecompressor().stream_reader(fh)
        with tarfile.open(fileobj=reader, mode="r|") as tar:
            tar.extractall(DATA)


ensure_data()
questions = []
for line in open(GOLD, encoding="utf-8"):
    if line.strip():
        questions.append(json.loads(line))
paper_ids = sorted({q["paper_id"] for q in questions})
print(len(questions), "questions over", len(paper_ids), "papers")

1852 questions over 137 papers


### Step 3 · Text normaliser (same as notebook 03)

In [3]:
import re
import unicodedata


def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"-\s*\n\s*", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


print(norm("Fourteen days."))

fourteen days


### Step 4 · Docling chunker, plain or labelled

In [4]:
from docling.chunking import HybridChunker
from docling_core.types.doc import DoclingDocument


def paper_title(doc):
    for item in doc.texts:
        label = getattr(item.label, "value", str(item.label))
        if label == "title" and item.text.strip():
            return item.text.strip()
    return doc.name


def load_doc(pid):
    return DoclingDocument.load_from_json(PARSED / pid / (pid + ".docling.json"))


def docling_chunks(doc, pid, max_tokens, labelled):
    chunker = HybridChunker(max_tokens=max_tokens)
    title = paper_title(doc)
    rows = []
    for ch in chunker.chunk(doc):
        headings = []
        if ch.meta.headings:
            for h in ch.meta.headings:
                headings.append(h)
        index_text = ch.text
        if labelled:
            index_text = "Paper: " + title + "\n" + chunker.contextualize(ch)
        rows.append({"paper": pid, "section": " > ".join(headings),
                     "text": ch.text, "index_text": index_text})
    return rows

### Step 5 · Chunk all papers: plain and labelled at 512, small 256-token chunks for Part C

In [5]:
import time

t0 = time.time()
docs = {}
plain_512 = []
label_512 = []
small_256 = []
for pid in paper_ids:
    doc = load_doc(pid)
    docs[pid] = doc
    plain_512.extend(docling_chunks(doc, pid, 512, labelled=False))
    label_512.extend(docling_chunks(doc, pid, 512, labelled=True))
    small_256.extend(docling_chunks(doc, pid, 256, labelled=True))
print(f"{time.time() - t0:.0f}s | plain_512 {len(plain_512)} | label_512 {len(label_512)} | small_256 {len(small_256)}")

372s | plain_512 5912 | label_512 5912 | small_256 10671


### Step 6 · Ranking and scoring functions

In [6]:
from rank_bm25 import BM25Okapi


def bm25_rank(chunks, k=50):
    """Top-k chunk indices per question with BM25 over index_text."""
    corpus = []
    for c in chunks:
        corpus.append(norm(c["index_text"]).split())
    bm25 = BM25Okapi(corpus)
    ranked = []
    for q in questions:
        scores = bm25.get_scores(norm(q["question"]).split())
        top = sorted(range(len(scores)), key=lambda j: -scores[j])[:k]
        ranked.append(top)
    return ranked

In [7]:
def hits_at(chunks, ranked, k=5, qs=None):
    """1 if a top-k chunk is from the question's paper and contains the gold answer."""
    if qs is None:
        qs = list(range(len(questions)))
    texts = []
    for c in chunks:
        texts.append(norm(c["text"]))
    out = []
    for qi in qs:
        q = questions[qi]
        ans = norm(q["gold_answer"])
        hit = 0
        for j in ranked[qi][:k]:
            if chunks[j]["paper"] == q["paper_id"] and ans and ans in texts[j]:
                hit = 1
                break
        out.append(hit)
    return out

In [8]:
import random


def paired_ci(a, b, reps=2000, seed=0):
    rng = random.Random(seed)
    n = len(a)
    diffs = []
    for _ in range(reps):
        s = 0
        for _ in range(n):
            k = rng.randrange(n)
            s += b[k] - a[k]
        diffs.append(s / n)
    diffs.sort()
    observed = sum(b[k] - a[k] for k in range(n)) / n
    return observed, diffs[int(0.025 * reps)], diffs[int(0.975 * reps)]


def report(name_a, a, name_b, b):
    d, lo, hi = paired_ci(a, b)
    verdict = "real gain" if lo > 0 else ("real loss" if hi < 0 else "tie (CI crosses 0)")
    print(f"{name_b} vs {name_a}: {d * 100:+.1f} pts  95% CI [{lo * 100:+.1f}, {hi * 100:+.1f}]  {verdict}")

### Step 7 · BM25 baseline for the two 512 rows (matches notebook 03)

In [9]:
bm25_plain = bm25_rank(plain_512)
bm25_label = bm25_rank(label_512)
h_bm25_plain = hits_at(plain_512, bm25_plain)
h_bm25_label = hits_at(label_512, bm25_label)
print(f"BM25 plain {sum(h_bm25_plain) / len(questions):.1%} | BM25 label {sum(h_bm25_label) / len(questions):.1%}")
report("bm25 plain", h_bm25_plain, "bm25 label", h_bm25_label)

BM25 plain 67.8% | BM25 label 68.2%


bm25 label vs bm25 plain: +0.4 pts  95% CI [-0.3, +1.1]  tie (CI crosses 0)


## Part A · Does the label help dense retrieval?

BM25 only matches words, so a label adds a few words. A dense model reads meaning, so the
paper title and section path might help it more. Same chunks, same questions, only the
retriever changes.

In [10]:
from sentence_transformers import SentenceTransformer


def dense_rank(chunks, model_name, query_prefix="", doc_prefix="", k=50, batch_size=32):
    """Top-k chunk indices per question by cosine similarity."""
    model = SentenceTransformer(model_name)
    doc_texts = []
    for c in chunks:
        doc_texts.append(doc_prefix + c["index_text"])
    q_texts = []
    for q in questions:
        q_texts.append(query_prefix + q["question"])
    D = model.encode(doc_texts, batch_size=batch_size, normalize_embeddings=True,
                     show_progress_bar=False, convert_to_tensor=True)
    Q = model.encode(q_texts, batch_size=batch_size, normalize_embeddings=True,
                     show_progress_bar=False, convert_to_tensor=True)
    sims = Q @ D.T
    top = sims.topk(k, dim=1).indices.tolist()
    return top

### Step 8 · Dense Recall@5, plain vs labelled (small CPU model)

In [11]:
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
t0 = time.time()
dense_plain = dense_rank(plain_512, DENSE_MODEL)
dense_label = dense_rank(label_512, DENSE_MODEL)
h_dense_plain = hits_at(plain_512, dense_plain)
h_dense_label = hits_at(label_512, dense_label)
print(f"{time.time() - t0:.0f}s | dense plain {sum(h_dense_plain) / len(questions):.1%} | dense label {sum(h_dense_label) / len(questions):.1%}")
report("dense plain", h_dense_plain, "dense label", h_dense_label)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

660s | dense plain 38.6% | dense label 37.5%


dense label vs dense plain: -1.0 pts  95% CI [-2.2, +0.2]  tie (CI crosses 0)


**YOUR TURN (Part A).** On Colab (free T4), set `DENSE_MODEL` to a stronger model and its
prefixes, then rerun Step 8 and add the fixed-512 chunks as a third row:

- `BAAI/bge-small-en-v1.5` with `query_prefix="Represent this sentence for searching relevant passages: "`
- `Qwen/Qwen3-Embedding-0.6B` (use the query instruction from its model card)

Question to answer: is the label gain bigger for dense than for BM25?

## Part B · Table questions only

A question counts as a table question when its gold answer appears inside one of its
paper's tables. That is where Docling repeats table headers, so it is where a difference
should show.

### Step 9 · Find table questions

In [12]:
table_text = {}
for pid in paper_ids:
    doc = docs[pid]
    parts = []
    for t in doc.tables:
        parts.append(norm(t.export_to_markdown(doc=doc)))
    table_text[pid] = " ".join(parts)


def is_table_question(q):
    ans = norm(q["gold_answer"])
    return bool(ans) and ans in table_text.get(q["paper_id"], "")


table_qs = []
for i in range(len(questions)):
    if is_table_question(questions[i]):
        table_qs.append(i)
print(len(table_qs), "table questions out of", len(questions))

60 table questions out of 1852


### Step 10 · BM25 Recall@5 on table questions only

In [13]:
t_plain = hits_at(plain_512, bm25_plain, qs=table_qs)
t_label = hits_at(label_512, bm25_label, qs=table_qs)
print(f"table questions | docling plain {sum(t_plain) / len(table_qs):.1%} | docling label {sum(t_label) / len(table_qs):.1%}")
report("plain (tables)", t_plain, "label (tables)", t_label)

table questions | docling plain 68.3% | docling label 70.0%
label (tables) vs plain (tables): +1.7 pts  95% CI [+0.0, +5.0]  tie (CI crosses 0)


**YOUR TURN (Part B).** Add the fixed-512 chunker from notebook 03 to this subset. If
the table subset is small, say so next to the number: a wide CI is a finding too.

## Part C · Parent-child (small-to-big)

Search with small 256-token chunks (more precise), then hand the LLM the **parent**: every
chunk of the same paper under the same section heading. A hit means the gold answer is in
one of the top-5 parents.

### Step 11 · Build parents: group small chunks by paper and section

In [14]:
parent_of = []
parent_text = {}
for c in small_256:
    key = (c["paper"], c["section"])
    parent_of.append(key)
    if key not in parent_text:
        parent_text[key] = []
    parent_text[key].append(c["text"])

parent_norm = {}
for key in parent_text:
    parent_norm[key] = norm(" ".join(parent_text[key]))
print(len(small_256), "small chunks grouped into", len(parent_norm), "parents")

10671 small chunks grouped into 2330 parents


### Step 12 · Parent-child Recall@5

In [15]:
def parent_hits(ranked, k=5):
    out = []
    for qi in range(len(questions)):
        q = questions[qi]
        ans = norm(q["gold_answer"])
        seen = []
        for j in ranked[qi]:
            key = parent_of[j]
            if key not in seen:
                seen.append(key)
            if len(seen) == k:
                break
        hit = 0
        for key in seen:
            if key[0] == q["paper_id"] and ans and ans in parent_norm[key]:
                hit = 1
                break
        out.append(hit)
    return out


bm25_small = bm25_rank(small_256)
h_small = hits_at(small_256, bm25_small)
h_parent = parent_hits(bm25_small)
print(f"small 256 alone {sum(h_small) / len(questions):.1%} | parent-child {sum(h_parent) / len(questions):.1%} | label 512 {sum(h_bm25_label) / len(questions):.1%}")
report("label 512", h_bm25_label, "parent-child", h_parent)

small 256 alone 65.0% | parent-child 72.2% | label 512 68.2%


parent-child vs label 512: +4.0 pts  95% CI [+2.8, +5.3]  real gain


**Read this carefully.** A parent is bigger than a 512-token chunk, so it contains more
answers by construction. Report the average parent size in tokens next to the Recall: a
gain that comes only from handing the LLM much more text is not free.

**YOUR TURN (Part C).** Print the average parent length, and try parents capped at about
1,000 tokens.

## Part D · Export the chunks for the retriever track

### Step 13 · Write `chunks.jsonl` (labelled Docling 512) into the gitignored data folder

In [16]:
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
out_path = CHUNK_DIR / "docling_512_label.jsonl"
with open(out_path, "w", encoding="utf-8") as fh:
    for i in range(len(label_512)):
        c = label_512[i]
        fh.write(json.dumps({"chunk_id": c["paper"] + ":" + str(i), "paper": c["paper"],
                             "section": c["section"], "text": c["text"],
                             "index_text": c["index_text"]}) + "\n")
print("wrote", out_path.relative_to(REPO), "|", len(label_512), "chunks")

wrote 04-benchmarks/covid-qa/data/chunks/docling_512_label.jsonl | 5912 chunks


## Known gaps

- The CPU model in Part A is small and old; it shows the wiring, not the best dense number.
- Table questions are found by text match against table markdown; an answer that also
  appears in prose is still counted as a table question.
- Parent size is not controlled yet (see Part C).